# Week 4, Lab 5 — Mini-project: human-in-the-loop graph


In [1]:
WEEK = 'Week 4'
LAB = 'Lab 5 — HITL mini-project'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


Week 4 / Lab 5 — HITL mini-project
Backend: ollama
Need Ollama running: `ollama serve` and `ollama pull llama3.2:1b`
If import failed, unzip/clone the WHOLE course folder (not a single notebook).


In [3]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

llm = get_langchain_llm()

class S(TypedDict):
    question: str
    route: str
    tool_result: str
    approved: bool
    answer: str

def router(state: S) -> S:
    r = llm.invoke(f"Classify as math or research (one word): {state['question']}").content.lower()
    return {"route": "math" if "math" in r else "research"}

def math_node(state: S) -> S:
    expr = llm.invoke(f"Extract arithmetic only: {state['question']}").content.strip()
    return { "tool_result": calculator(expr)}

def research_node(state: S) -> S:
    topic = llm.invoke(f"Extract the topic keyword: {state['question']}").content.strip()
    return {"tool_result": lookup_fact(topic)}

def hitl(state: S) -> S:
    print("TOOL RESULT:", state["tool_result"])
    yn = (input("Approve this tool result? [y/n]: ").strip().lower() or "y")
    return {"approved": yn.startswith("y")}

def responder(state: S) -> S:
    if not state.get("approved"):
        return {"answer": "Stopped: human rejected the tool result."}
    ans = llm.invoke(f"Question: {state['question']}\nTool: {state['tool_result']}\nShort answer:").content
    return {"answer": ans}

def after_router(state: S) -> Literal["math", "research"]:
    return "math" if state["route"] == "math" else "research"

g = StateGraph(S)
for n, fn in [("router", router), ("math", math_node), ("research", research_node), ("hitl", hitl), ("responder", responder)]:
    g.add_node(n, fn)
g.add_edge(START, "router")
g.add_conditional_edges("router", after_router, {"math": "math", "research": "research"})
g.add_edge("math", "hitl")
g.add_edge("research", "hitl")
g.add_edge("hitl", "responder")
g.add_edge("responder", END)
app = g.compile()

demo = app.invoke({"question": "What is Langchain", "route": "", "tool_result": "", "approved": False, "answer": ""})
print("ANSWER:", demo["answer"])


TOOL RESULT: LangChain is a toolkit for prompts, chains, retrievers, and tool-using agents.
ANSWER: LangChain is a toolkit designed for working with prompts, chains, retrievers, and tools to create agent systems.


On Colab, `input()` works. For a silent demo, auto-set `approved=True` in `hitl`.
